In [12]:
# import
import pandas as pd
import numpy as np
from pathlib import Path
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent      # if notebook is inside notebooks/
DATA_DIR = PROJECT_ROOT / "data"

In [13]:
customer_database = pd.read_csv(DATA_DIR / "processed/base_X_test.csv") # getting the Test data
print(customer_database.shape)
y_test = pd.read_csv(DATA_DIR / "processed/base_y_test.csv").squeeze() # Getting the Test Target Variables
print(y_test.shape)

(10243, 56)
(10243,)


### 1. ***Generating Customer Database for Streamlit UI***

In [14]:
# Generate banking-style Applicant IDs
customer_database.insert(
    0,
    "PROSPECTID",
    [f"CUST{i+1}" for i in range(len(customer_database))]
)

print(customer_database.head())

  PROSPECTID  pct_tl_open_L6M  pct_tl_closed_L6M  Tot_TL_closed_L12M  \
0      CUST1            0.300              0.300                   4   
1      CUST2            0.222              0.222                   4   
2      CUST3            0.000              0.000                   0   
3      CUST4            0.143              0.071                   2   
4      CUST5            1.000              0.000                   0   

   pct_tl_open_L12M  pct_tl_closed_L12M  Tot_Missed_Pmnt  CC_TL  Home_TL  \
0             0.500               0.400                2      0        0   
1             0.333               0.444                0      0        0   
2             1.000               0.000                0      0        0   
3             0.214               0.143                0      0        0   
4             1.000               0.000                1      0        0   

   PL_TL  ...  first_prod_enq2_ConsumerLoan  first_prod_enq2_HL  \
0      0  ...                             1

In [15]:
hidden_ground_truth = pd.DataFrame({
    "PROSPECTID": customer_database["PROSPECTID"],
    "Actual_Risk_Grade": y_test.values
})

In [16]:
customer_database.to_csv(DATA_DIR / "customer_database.csv",index = False)
hidden_ground_truth.to_csv(DATA_DIR / "hidden_ground_truth.csv", index= False)

### Making a Readable Database For Customer Profiling

In [17]:
DROP_COLUMNS = [

    "pct_tl_open_L6M",
    "pct_tl_closed_L6M",
    "Tot_TL_closed_L12M",
    "pct_tl_open_L12M",
    "pct_tl_closed_L12M",

    "num_deliq_6_12mts",
    "max_deliq_6mts",
    "max_deliq_12mts",

    "num_std_12mts",
    "num_sub",
    "num_sub_12mts",
    "num_dbt",
    "num_dbt_12mts",
    "num_lss",

    "CC_Flag",
    "PL_Flag",
    "HL_Flag",
    "GL_Flag",

    "pct_PL_enq_L6m_of_ever",
    "pct_CC_enq_L6m_of_ever",

    "first_prod_enq2_AL",
    "first_prod_enq2_CC",
    "first_prod_enq2_ConsumerLoan",
    "first_prod_enq2_HL",
    "first_prod_enq2_PL",
    "first_prod_enq2_others",

    "last_prod_enq2_AL",
    "last_prod_enq2_CC",
    "last_prod_enq2_ConsumerLoan",
    "last_prod_enq2_HL",
    "last_prod_enq2_PL",
    "last_prod_enq2_others"
]

In [18]:
customer_profile = customer_database.drop(DROP_COLUMNS,axis=1)
customer_profile.shape # keep Only 25 columns for profiling

(10243, 25)

In [19]:
# Binary Decoding

customer_profile["GENDER"] = customer_profile["GENDER"].map({
    1:"Male",
    0: "Female"
})

customer_profile["MARITALSTATUS"] = customer_profile["MARITALSTATUS"].map({
    1 : "Married",
    0 : "Single"
})

In [20]:
customer_profile[['GENDER','MARITALSTATUS']]

,GENDER,MARITALSTATUS
0,Male,Married
1,Male,Married
2,Male,Single
3,Female,Married
4,Male,Single
...,...,...
10238,Male,Married
10239,Male,Married
10240,Male,Married
10241,Male,Single


### Decoding Education 

In [21]:
education_mapping = {
    -1:"OTHERS",
    0:"SSC",
    1:"12TH",
    2:"UNDER GRADUATE",
    3:"GRADUATE",
    4:"POST-GRADUATE",
    5:"PROFESSIONAL"
}

customer_profile["EDUCATION"] = customer_profile["EDUCATION"].map(education_mapping)

In [22]:
customer_profile['EDUCATION']

0                  12TH
1              GRADUATE
2              GRADUATE
3              GRADUATE
4              GRADUATE
              ...      
10238          GRADUATE
10239              12TH
10240    UNDER GRADUATE
10241              12TH
10242     POST-GRADUATE
Name: EDUCATION, Length: 10243, dtype: object

### Renaming Columns For Readability

In [23]:
COLUMN_MAPPING = {

    # Identity
    "PROSPECTID": "Applicant ID",

    # Customer Information
    "GENDER": "Gender",
    "MARITALSTATUS": "Marital Status",
    "EDUCATION": "Education",
    "NETMONTHLYINCOME": "Monthly Income (₹)",
    "Time_With_Curr_Empr": "Employment Duration (Months)",

    # Loan Portfolio
    "CC_TL": "Credit Card Accounts",
    "PL_TL": "Personal Loan Accounts",
    "Home_TL": "Home Loan Accounts",
    "Secured_TL": "Secured Loan Accounts",
    "Unsecured_TL": "Unsecured Loan Accounts",
    "Other_TL": "Other Loan Accounts",

    # Credit History
    "Age_Oldest_TL": "Oldest Credit Line (Months)",
    "Age_Newest_TL": "Newest Credit Line (Months)",

    "time_since_recent_payment": "Months Since Last Payment",
    "time_since_recent_enq": "Months Since Last Credit Enquiry",

    # Delinquency
    "Tot_Missed_Pmnt": "Total Missed Payments",
    "recent_level_of_deliq": "Current Delinquency Level",
    "max_recent_level_of_deliq": "Highest Recent Delinquency",
    "num_times_60p_dpd": "60+ DPD Occurrences",

    # Enquiries
    "enq_L3m": "Credit Enquiries (Last 3 Months)",
    "CC_enq_L12m": "Credit Card Enquiries (12 Months)",
    "PL_enq_L12m": "Personal Loan Enquiries (12 Months)",

    # Exposure
    "pct_currentBal_all_TL": "Current Balance Ratio (%)",
    "max_unsec_exposure_inPct": "Maximum Unsecured Exposure (%)"
}

In [24]:
customer_profile.rename(
    columns=COLUMN_MAPPING,
    inplace=True
)

In [25]:
customer_profile.columns

Index(['Applicant ID', 'Total Missed Payments', 'Credit Card Accounts',
       'Home Loan Accounts', 'Personal Loan Accounts', 'Secured Loan Accounts',
       'Unsecured Loan Accounts', 'Other Loan Accounts',
       'Oldest Credit Line (Months)', 'Newest Credit Line (Months)',
       'Months Since Last Payment', 'Highest Recent Delinquency',
       '60+ DPD Occurrences', 'Current Delinquency Level',
       'Credit Card Enquiries (12 Months)',
       'Personal Loan Enquiries (12 Months)',
       'Months Since Last Credit Enquiry', 'Credit Enquiries (Last 3 Months)',
       'Monthly Income (₹)', 'Employment Duration (Months)',
       'Current Balance Ratio (%)', 'Maximum Unsecured Exposure (%)',
       'Marital Status', 'Education', 'Gender'],
      dtype='object')

**Converting Employment Duration Into Years**

In [26]:
customer_profile["Employment Duration (Months)"] = (
    customer_profile["Employment Duration (Months)"] / 12
).round(1).astype(str) + " Years"

**Money**

In [27]:
customer_profile["Monthly Income (₹)"] = (
    customer_profile["Monthly Income (₹)"]
    .apply(lambda x: f"₹{x:,.0f}")
)

**Percentage Columns**

In [28]:
PERCENT_COLUMNS = [
    "Current Balance Ratio (%)",
    "Maximum Unsecured Exposure (%)"
]

for col in PERCENT_COLUMNS:
    customer_profile[col] = (
        customer_profile[col]
        .round(1)
        .astype(str)
        + "%"
    )

In [29]:
customer_profile.fillna("Not Available", inplace=True)

In [30]:
customer_profile['Current Delinquency Level']

0         0
1         5
2         0
3        26
4         0
         ..
10238    30
10239     0
10240    25
10241    28
10242     0
Name: Current Delinquency Level, Length: 10243, dtype: int64

In [31]:
customer_profile["Total Active Loan Accounts"] = (
    customer_profile["Credit Card Accounts"] +
    customer_profile["Personal Loan Accounts"] +
    customer_profile["Home Loan Accounts"] +
    customer_profile["Other Loan Accounts"]
)

In [32]:
customer_profile.to_csv(DATA_DIR / "customer_profile.csv",index=False)

In [33]:
customer_profile.head()

,Applicant ID,Total Missed Payments,Credit Card Accounts,Home Loan Accounts,Personal Loan Accounts,Secured Loan Accounts,Unsecured Loan Accounts,Other Loan Accounts,Oldest Credit Line (Months),Newest Credit Line (Months),...,Months Since Last Credit Enquiry,Credit Enquiries (Last 3 Months),Monthly Income (₹),Employment Duration (Months),Current Balance Ratio (%),Maximum Unsecured Exposure (%),Marital Status,Education,Gender,Total Active Loan Accounts
0,CUST1,2,0,0,0,2,8,2,35.0,2.0,...,42.0,6.0,"₹30,000",15.0 Years,0.9%,1.3%,Married,12TH,Male,2
1,CUST2,0,0,0,0,8,1,0,33.0,4.0,...,124.0,0.0,"₹16,000",5.5 Years,0.7%,2.4%,Married,GRADUATE,Male,0
2,CUST3,0,0,0,0,1,1,1,9.0,9.0,...,0.0,2.0,"₹18,000",6.5 Years,1.4%,0.1%,Single,GRADUATE,Male,1
3,CUST4,0,0,0,0,5,9,2,71.0,3.0,...,102.0,0.0,"₹40,000",50.6 Years,0.7%,4.3%,Married,GRADUATE,Female,2
4,CUST5,1,0,0,0,0,2,1,5.0,5.0,...,166.0,0.0,"₹25,000",30.7 Years,0.7%,1.0%,Single,GRADUATE,Male,1


In [34]:
customer_database.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10243 entries, 0 to 10242
Data columns (total 57 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   PROSPECTID                    10243 non-null  object 
 1   pct_tl_open_L6M               10243 non-null  float64
 2   pct_tl_closed_L6M             10243 non-null  float64
 3   Tot_TL_closed_L12M            10243 non-null  int64  
 4   pct_tl_open_L12M              10243 non-null  float64
 5   pct_tl_closed_L12M            10243 non-null  float64
 6   Tot_Missed_Pmnt               10243 non-null  int64  
 7   CC_TL                         10243 non-null  int64  
 8   Home_TL                       10243 non-null  int64  
 9   PL_TL                         10243 non-null  int64  
 10  Secured_TL                    10243 non-null  int64  
 11  Unsecured_TL                  10243 non-null  int64  
 12  Other_TL                      10243 non-null  int64  
 13  A